# Entorno de Almacén Personalizado - Demostración y Pruebas

Este notebook sirve para demostrar y probar el entorno personalizado de Gymnasium `WarehouseEnv` que simula un almacén automatizado de 5x5.

El robot debe recoger un paquete en la zona de **PICKUP** y llevarlo a la zona de **DROP-OFF** esquivando un **Obstáculo** móvil estocástico.

In [ ]:
# Habilitar recarga automática de módulos modificados
%load_ext autoreload
%autoreload 2

import time
import gymnasium as gym
from warehouse_env import WarehouseEnv

## 1. Prueba en Modo Consola (`render_mode='ansi'`)

Primero instanciamos y visualizamos el entorno en formato de texto sencillo para validar que los estados iniciales se carguen correctamente y las coordenadas se mapeen de forma correcta.

In [ ]:
# Crear entorno en modo texto
env_text = WarehouseEnv(render_mode="ansi", max_steps=50)

# Resetear el entorno
obs, info = env_text.reset()

print("=== Estado Inicial ===")
print(f"Observación: {obs}")
print("Explicación del vector de estado:")
print(f" - Robot Pos: ({obs[0]}, {obs[1]})")
print(f" - Obstáculo Pos: ({obs[2]}, {obs[3]})")
print(f" - Pickup Pos: ({obs[4]}, {obs[5]})")
print(f" - Dropoff Pos: ({obs[6]}, {obs[7]})")
print(f" - Cargando Paquete: {'Sí' if obs[8] == 1 else 'No'}")
print(f"Info Adicional: {info}")

# Dibujar entorno en texto
env_text.render()

### Ejecutar unos pasos aleatorios en consola

Probemos el paso a paso del robot seleccionando acciones de manera aleatoria. Presta atención a las penalizaciones de paso (-1) y cómo se actualizan las posiciones.

In [ ]:
print("Ejecutando pasos de prueba aleatorios...")
for step in range(5):
    action = env_text.action_space.sample()
    
    # Mapeo de nombres de acciones para depuración
    action_names = {0: "ARRIBA", 1: "ABAJO", 2: "IZQUIERDA", 3: "DERECHA", 4: "RECOGER (PICKUP)", 5: "ENTREGAR (DROPOFF)"}
    
    obs, reward, terminated, truncated, info = env_text.step(action)
    
    print(f"\n--- Paso {step + 1} ---")
    print(f"Acción tomada: {action} ({action_names[action]})")
    print(f"Recompensa: {reward}")
    print(f"Terminado: {terminated} | Truncado: {truncated}")
    print(f"Estado de Carga: {obs[8]}")
    
    env_text.render()
    
    if terminated or truncated:
        print("Episodio finalizado durante la prueba.")
        break
        
env_text.close()

## 2. Demostración Visual Premium con Pygame (`render_mode='human'`)

Ahora ejecutaremos un bucle de simulación con la renderización visual en una ventana externa de Pygame. 

El robot se moverá de forma aleatoria, el obstáculo se moverá estocásticamente, y el bucle simulará 3 episodios completos. 

**Nota:** Si el robot colisiona con el obstáculo rojo o logra recoger y entregar el paquete exitosamente, el episodio se reiniciará automáticamente.

In [ ]:
# Instanciar en modo visual
env_visual = WarehouseEnv(render_mode="human", max_steps=150)

num_episodes = 3
print("Abriendo ventana de Pygame. Presiona detener en el notebook para finalizar.")

try:
    for episode in range(num_episodes):
        obs, info = env_visual.reset()
        terminated = False
        truncated = False
        total_reward = 0
        steps = 0
        
        print(f"\n--- Iniciando Episodio {episode + 1} ---")
        
        while not (terminated or truncated):
            # 1. Tomar acción aleatoria
            action = env_visual.action_space.sample()
            
            # Si el robot está sobre el pickup y no tiene paquete, forzar intentar recoger (acción 4) 
            # con cierta probabilidad para mostrar la física y gráfica de carga en la demo aleatoria.
            rx, ry = obs[0], obs[1]
            px, py = obs[4], obs[5]
            dx, dy = obs[6], obs[7]
            carrying = obs[8]
            
            if rx == px and ry == py and carrying == 0:
                action = 4 # Forzar Pickup si coincide para que el estudiante vea cómo carga
            elif rx == dx and ry == dy and carrying == 1:
                action = 5 # Forzar Drop-off si coincide y lleva carga
            
            # 2. Aplicar acción en el entorno
            obs, reward, terminated, truncated, info = env_visual.step(action)
            total_reward += reward
            steps += 1
            
            # 3. Pequeña espera para poder seguir visualmente la animación (Pygame corre a 4 fps de base)
            time.sleep(0.1)
            
            if steps % 15 == 0:
                print(f"Episodio {episode+1} | Paso: {steps} | Recompensa acumulada: {total_reward}")
                
        # Informar resultado del episodio
        print(f"Episodio {episode + 1} finalizado en {steps} pasos.")
        print(f"Recompensa final acumulada: {total_reward}")
        if terminated:
            if total_reward > 0:
                print("¡Éxito! El robot entregó el paquete exitosamente.")
            else:
                print("¡Accidente! El robot colisionó con el obstáculo en la posición:", obs[2:4])
        elif truncated:
            print("¡Límite de tiempo! Se alcanzaron los pasos máximos sin finalizar.")
            
        # Pausa antes del siguiente episodio
        time.sleep(1.5)

except KeyboardInterrupt:
    print("\nSimulación interrumpida por el usuario.")
finally:
    env_visual.close()
    print("Entorno cerrado correctamente.")